# FILIP GUARD — un singur apel AI, după expirarea limitei

Păstrează cheia existentă `GROQ_API_KEY` în Colab Secrets. Folosește CPU gratuit, fără card, Pro sau abonamente.

Se verifică doar cazul `craft`, nu întreaga suită. Dacă limita furnizorului sau bugetul local sunt active, nu se trimite nicio cerere și raportul arată motivul/termenul. Nu șterge baza de date și nu reporni sesiunea pentru a ocoli limitele.

O veche eroare 429 fără termen păstrat primește o așteptare conservatoare de 24h de la cererea originală, plus 20 secunde marjă de durată, marcată ca termen local estimat, nu expirare confirmată de Groq. Termenele indicate de furnizor nu sunt scurtate. După termen se poate încerca manual cel mult un apel; un nou 429 blochează din nou, fără retry.

Rulează celulele și descarcă JSON-ul. Nu trimite cheia în chat sau GitHub. Testarea completă urmează numai după un răspuns real nou acceptat.


In [ ]:
import os, pathlib, subprocess, sys, re, sqlite3
os.umask(0o077)
repository='https://github.com/Ciobanu87/filip-guard.git'
branch='filip-guard-development'
remote=subprocess.check_output(['git','ls-remote',repository,'refs/heads/'+branch],text=True).split()[0]
if not re.fullmatch('[0-9a-f]{40}',remote):raise RuntimeError('Development revision could not be verified')
checkout=pathlib.Path('/content/filip-guard-ai-validation-'+remote[:12])
if not checkout.exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',branch,repository,str(checkout)],check=True,capture_output=True)
revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd=checkout,text=True).strip()
if revision!=remote:raise RuntimeError('Development branch changed during setup; rerun this setup cell')
print('Repository development revision:',revision)
subprocess.run([sys.executable,'-m','pip','install','-q','beautifulsoup4>=4.12,<5','cryptography>=44,<49','requests>=2.32,<3'],check=True)
# A notebook kernel can retain old Python modules even after a new clone.
for name,module in list(sys.modules.items()):
    origin=getattr(module,'__file__',None)
    if isinstance(origin,str) and origin.startswith('/content/filip-guard-ai-validation'):
        del sys.modules[name]
sys.path[:]=[entry for entry in sys.path if not str(entry).startswith('/content/filip-guard-ai-validation')]
sys.path.insert(0,str(checkout))
# Share the test budget across source revisions; do not erase old reservations.
state=pathlib.Path('/content/filip-guard-cloud-validation-state');state.mkdir(exist_ok=True,mode=0o700)
ledger=state/'ledger.sqlite'
legacy=pathlib.Path('/content/filip-guard-ai-validation/validation-ledger.sqlite')
if not ledger.exists() and legacy.is_file():
    source=sqlite3.connect(legacy.resolve().as_uri()+'?mode=ro',uri=True)
    target=sqlite3.connect(ledger)
    try:source.backup(target)
    finally:source.close();target.close()
    ledger.chmod(0o600)


In [ ]:
#@title Confirmări obligatorii — nu activa plăți
free_organization_without_billing = False #@param {type:"boolean"}
synthetic_data_transfer_authorized = False #@param {type:"boolean"}
one_live_request_authorized = False #@param {type:"boolean"}
if not all((free_organization_without_billing,synthetic_data_transfer_authorized,one_live_request_authorized)):
    raise RuntimeError('Confirmă organizația Free fără facturare și testele sintetice înainte de rulare.')
from google.colab import userdata
from advisor_groq import GroqFreeProvider
from contextual_advisor import ContextualAdvisor
from validate_cloud_advisor import run_probe
import re
try:
    _key_check=userdata.get('GROQ_API_KEY')
    if not isinstance(_key_check,str) or not re.fullmatch(r'gsk_[A-Za-z0-9_-]{16,200}',_key_check):raise ValueError()
except Exception:
    raise RuntimeError('Adaugă GROQ_API_KEY în Colab Secrets și permite accesul notebookului. Nu lipi cheia în cod.') from None
finally:
    _key_check=None
provider=GroqFreeProvider(lambda:userdata.get('GROQ_API_KEY'),free_plan_confirmed=True,data_consent=True)
advisor=ContextualAdvisor(ledger,provider,daily_calls=20,timeout=20,max_output_tokens=256)
print('Configurație pregătită. Nicio inferență nu a fost executată de această celulă.')


In [ ]:
import json
report=run_probe(advisor,live=True,progress=lambda result:print(json.dumps(result,ensure_ascii=False),flush=True))
report['repository_revision']=revision
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Inferență reală nouă:',report['real_inference_executed'])
print('Rezultate AI acceptate:',report['accepted_ai_cases'])
print('Calitatea necesită evaluare umană:',not report['quality_validated'])
print('Următorul termen UTC:',report.get('next_eligible_utc'))
print('Termen confirmat de furnizor:',report.get('provider_expiry_confirmed'))
print('Apeluri furnizor încercate:',report['attempted_cases'])


In [ ]:
import uuid
from google.colab import files
report_path=checkout/('cloud-validation-'+uuid.uuid4().hex+'.json')
with report_path.open('x',encoding='utf-8') as out:
    json.dump(report,out,ensure_ascii=False,indent=2)
report_path.chmod(0o600)
files.download(str(report_path))
